# Sales Data Consolidation and Validation

This notebook represents the second stage of the Sales ETL Pipeline.

The objective is to consolidate all detailed sales reports extracted from the sales platform and validate them against a second source containing active sales documents.

This process removes cancelled transactions and prepares a reliable dataset for later cleaning, reporting and analysis.

## 1. Project Context

The previous extraction stage generated multiple Excel files containing detailed sales transactions by product.

However, the detailed report alone does not guarantee that every sales document is still active. Some documents may have been cancelled after being issued.

To solve this, a second report containing valid or non-cancelled sales documents is used as a validation source.

Main tasks:

1. Locate all extracted Excel reports.
2. Load and consolidate them into a single raw dataset.
3. Load the active document report.
4. Normalize the common key: `Folio`.
5. Keep only transactions linked to active documents.
6. Remove exact duplicates.
7. Export the validated dataset.
8. Generate a quality control summary.

## 2. Libraries and Tools

This notebook uses Python for data consolidation and validation.

Main tools:

- **Pandas:** data ingestion, transformation and validation.
- **Pathlib:** portable file and folder management.
- **Excel files:** input and output format used by the business system.

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 3. Configuration

This section defines the main project paths.

The notebook uses relative paths instead of local computer paths. This makes the project easier to understand and reuse from GitHub.

Expected structure:

```text
sales-etl-pipeline/
├── notebooks/
├── data/
│   ├── raw_reports/
│   └── validation/
└── outputs/
```

Real business files are not included in the repository.

In [ ]:
# ============================================================
# Configuration
# ============================================================

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_REPORTS_DIR = DATA_DIR / "raw_reports"
VALIDATION_DIR = DATA_DIR / "validation"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

ACTIVE_DOCUMENTS_FILE = VALIDATION_DIR / "active_sales_documents.xlsx"

RAW_CONSOLIDATED_FILE = OUTPUTS_DIR / "sales_detail_raw_consolidated.xlsx"
FINAL_OUTPUT_FILE = OUTPUTS_DIR / "sales_detail_validated.xlsx"

EXPORT_RAW_CONSOLIDATED = True

OUTPUTS_DIR.mkdir(exist_ok=True)

## 4. File Discovery

This step searches recursively for all Excel files generated during the extraction stage.

Temporary Excel files are ignored to avoid loading incomplete or system-generated files.

In [ ]:
# ============================================================
# File discovery
# ============================================================

excel_files = [
    file for file in RAW_REPORTS_DIR.rglob("*")
    if file.suffix.lower() in [".xlsx", ".xls"]
    and not file.name.startswith("~$")
]

print(f"Files found: {len(excel_files)}")

if not excel_files:
    raise FileNotFoundError(f"No Excel files were found in: {RAW_REPORTS_DIR}")

for file in excel_files[:10]:
    print(f"- {file}")

if len(excel_files) > 10:
    print("...")

## 5. File Ingestion

Each Excel file is loaded individually.

During this process:

- Row counts are collected.
- Original file information is preserved.
- Loading errors are registered.
- A file-level summary is generated for quality control.

In [ ]:
# ============================================================
# File ingestion
# ============================================================

dataframes = []
loading_errors = []
file_summary = []

for file in excel_files:
    try:
        df = pd.read_excel(file)

        row_count = len(df)
        column_names = list(df.columns)

        df["source_file"] = str(file)
        dataframes.append(df)

        file_summary.append({
            "file": str(file),
            "rows": row_count,
            "columns": ", ".join(map(str, column_names))
        })

        print(f"[OK] {file.name}: {row_count} rows")

    except Exception as error:
        loading_errors.append({
            "file": str(file),
            "error": str(error)
        })

        print(f"[ERROR] {file.name}: {error}")

if not dataframes:
    raise ValueError("No Excel files could be loaded successfully.")

df_file_summary = pd.DataFrame(file_summary)
df_loading_errors = pd.DataFrame(loading_errors)

## 6. Raw Dataset Consolidation

All successfully loaded files are concatenated into a single raw dataset.

At this stage, no business validation is applied yet. The objective is to create one consolidated table from all extracted reports.

In [ ]:
# ============================================================
# Raw dataset consolidation
# ============================================================

df_raw_sales = pd.concat(dataframes, ignore_index=True)

print(f"Total consolidated rows: {len(df_raw_sales)}")
print(f"Total detected columns: {len(df_raw_sales.columns)}")

display(df_raw_sales.head())

## 7. Optional Raw Export

The raw consolidated dataset can be exported before applying validation rules.

This is useful for auditability because it preserves the intermediate dataset generated directly from the extracted reports.

In [ ]:
# ============================================================
# Optional raw export
# ============================================================

if EXPORT_RAW_CONSOLIDATED:
    df_raw_sales.to_excel(RAW_CONSOLIDATED_FILE, index=False)
    print(f"[✓] Raw consolidated dataset exported to: {RAW_CONSOLIDATED_FILE}")
else:
    print("Raw consolidated export skipped.")

## 8. Load Active Document Report

This step loads the validation source containing active or non-cancelled sales documents.

The `Folio` column is used as the common key between the detailed sales reports and the active document report.

In [ ]:
# ============================================================
# Load active document report
# ============================================================

if not ACTIVE_DOCUMENTS_FILE.exists():
    raise FileNotFoundError(f"Validation file not found: {ACTIVE_DOCUMENTS_FILE}")

df_active_documents = pd.read_excel(ACTIVE_DOCUMENTS_FILE)

print(f"Rows in active document report: {len(df_active_documents)}")
display(df_active_documents.head())

## 9. Folio Validation and Normalization

Before filtering transactions, the `Folio` column is validated and normalized in both datasets.

This step ensures that the matching process is consistent even if folio values are stored with different formats.

In [ ]:
# ============================================================
# Folio validation and normalization
# ============================================================

if "Folio" not in df_raw_sales.columns:
    raise KeyError("The column 'Folio' does not exist in the consolidated sales dataset.")

if "Folio" not in df_active_documents.columns:
    raise KeyError("The column 'Folio' does not exist in the active document report.")

df_raw_sales["Folio"] = df_raw_sales["Folio"].astype(str).str.strip()
df_active_documents["Folio"] = df_active_documents["Folio"].astype(str).str.strip()

print("[✓] Folio normalization completed")

## 10. Active Transaction Filtering

The detailed sales dataset is filtered using the list of active document folios.

Only records whose `Folio` exists in the active document report are retained.

This removes transactions associated with cancelled documents.

In [ ]:
# ============================================================
# Active transaction filtering
# ============================================================

active_folios = set(df_active_documents["Folio"].dropna().unique())

df_valid_sales = df_raw_sales[df_raw_sales["Folio"].isin(active_folios)].copy()

print(f"Rows before validation: {len(df_raw_sales)}")
print(f"Unique active folios: {len(active_folios)}")
print(f"Rows after validation: {len(df_valid_sales)}")
print(f"Rows removed: {len(df_raw_sales) - len(df_valid_sales)}")

## 11. Final Cleaning

Basic cleaning operations are applied after validation.

Cleaning tasks:

- Convert the sales date column when available.
- Sort records by sales date.
- Remove exact duplicate rows.
- Preserve the validated structure for export.

In [ ]:
# ============================================================
# Final cleaning
# ============================================================

rows_before_duplicates = len(df_valid_sales)

if "Fecha Venta" in df_valid_sales.columns:
    df_valid_sales["Fecha Venta"] = pd.to_datetime(
        df_valid_sales["Fecha Venta"],
        errors="coerce",
        dayfirst=True
    )

    df_valid_sales = df_valid_sales.sort_values(
        by="Fecha Venta",
        ascending=True
    )

    print("[✓] Dataset sorted by Fecha Venta")
else:
    print("[!] Column 'Fecha Venta' was not found. Sorting step skipped.")

df_valid_sales = df_valid_sales.drop_duplicates()

rows_after_duplicates = len(df_valid_sales)
duplicates_removed = rows_before_duplicates - rows_after_duplicates

print(f"Exact duplicates removed: {duplicates_removed}")
print(f"Final rows: {len(df_valid_sales)}")

display(df_valid_sales.head())

## 12. Final Dataset Export

The validated dataset is exported to Excel.

This file becomes the main output of this ETL stage and can be used in later data cleaning, analytics, dashboarding or machine learning projects.

In [ ]:
# ============================================================
# Final dataset export
# ============================================================

df_valid_sales.to_excel(FINAL_OUTPUT_FILE, index=False)

print(f"[✓] Final validated dataset exported to: {FINAL_OUTPUT_FILE}")
print(f"Rows exported: {len(df_valid_sales)}")

## 13. Loading Error Report

This section displays files that could not be loaded correctly.

If the report is empty, all files were processed successfully.

In [ ]:
# ============================================================
# Loading error report
# ============================================================

if df_loading_errors.empty:
    print("[✓] All files were loaded successfully.")
else:
    print("[!] Some files could not be loaded:")
    display(df_loading_errors)

## 14. Quality Control Summary

This summary provides a quick overview of the consolidation and validation process.

It helps evaluate:

- Number of files found.
- Number of files loaded successfully.
- Number of loading errors.
- Number of rows before validation.
- Number of rows after validation.
- Number of duplicate records removed.

In [ ]:
# ============================================================
# Quality control summary
# ============================================================

quality_summary = pd.DataFrame([
    {
        "files_found": len(excel_files),
        "files_loaded_successfully": len(df_file_summary),
        "files_with_errors": len(df_loading_errors),
        "rows_consolidated": len(df_raw_sales),
        "unique_active_folios": len(active_folios),
        "rows_after_validation": len(df_valid_sales),
        "duplicates_removed": duplicates_removed
    }
])

display(quality_summary)

## 15. Output

Generated outputs:

```text
outputs/
├── sales_detail_raw_consolidated.xlsx
└── sales_detail_validated.xlsx
```

Dataset characteristics:

- Consolidated from multiple extracted Excel reports.
- Validated against active sales documents.
- Cancelled transactions removed.
- Exact duplicates removed.
- Ready for the next ETL stage.
```

This dataset is the foundation for the following stages of the project:

- Data cleaning and standardization.
- Final dataset export.
- Business intelligence reporting.
- Customer, product and revenue analysis.